**In-Class Exercise:** Double/Debiased Machine Learning (DML)

**Course:** ECON6083 - Machine Learning in Economics
**Topic:** Understanding DML's Two Pillars - Orthogonalization & Sample Splitting

---

## Part I: Conceptual Understanding

### Question 1: What does 'Double' in DML mean?

The 'Double' refers to using **TWO** nuisance models:
- $l(X) = E[Y|X]$: outcome model
- $m(X) = E[D|X]$: treatment model (propensity score)

**Why two models?** Because we need to partialling out confounders $X$ from **BOTH** $Y$ and $D$.

---

## Setup: Import Libraries and Generate Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
np.random.seed(42)
n = 500

# Generate confounders
age = np.random.normal(45, 10, n)
income = np.random.normal(50000, 20000, n)
education = np.random.normal(14, 3, n)
married = np.random.binomial(1, 0.6, n)
fam_size = np.random.poisson(2.5, n) + 1

X = pd.DataFrame({
    'X1': age,
    'X2': income / 10000,
    'X3': education,
    'X4': married,
    'X5': fam_size
})

# Generate treatment (401k participation)
propensity_logit = 0.5 * (age - 45)/10 + 0.3 * (income - 50000)/20000 - 0.2 * (education - 14)/3
D = (np.random.uniform(0, 1, n) < 1 / (1 + np.exp(-propensity_logit))).astype(int)

# Generate outcome: Y = 5000 + 2500*D + confounder effects + noise
# True ATE = $2,500
Y = 5000 + 2500 * D + 500 * age + 1000 * (income / 10000) + np.random.normal(0, 2000, n)

print(f"Data generated: N={n}, True ATE=$2,500")
print(f"401(k) participation rate: {D.mean():.1%}")


## Step 1: Split Sample for Cross-Fitting

DML requires sample splitting: train nuisance models on one part, predict on the other.

In [ ]:
# Split sample in half
sample1_idx, sample2_idx = ____  # TODO: train_test_split(np.arange(len(X)), test_size=0.5, random_state=42)

X1, X2 = X.iloc[sample1_idx], X.iloc[sample2_idx]
D1, D2 = D[sample1_idx], D[sample2_idx]
Y1, Y2 = Y[sample1_idx], Y[sample2_idx]

print(f"Sample 1: n={len(X1)}, Sample 2: n={len(X2)}")


## Part II: Comparison 1 - Why Orthogonalization Matters

We compare two approaches (both use sample splitting):
- **Naive ML**: Only residualize $Y$ (single nuisance)
- **Full DML**: Residualize **BOTH** $Y$ and $D$ (correct!)

**Task**: Train nuisance models on Sample 1, predict on Sample 2

In [ ]:
# Train nuisance models on Sample 1

# Outcome model: l(X) = E[Y|X]
l_model = RandomForestRegressor(
    n_estimators=50, max_depth=____, random_state=42  # TODO: 5
)
l_model.fit(X1, Y1)
Y2_hat = l_model.predict(X2)

# Treatment model: m(X) = E[D|X] (propensity score)
m_model = RandomForestRegressor(
    n_estimators=50, max_depth=____, random_state=42  # TODO: 5
)
m_model.fit(X1, D1)
D2_hat = m_model.predict(X2)


In [ ]:
# Method A: Naive ML (ONLY residualize Y, use raw D)
# This is WRONG because D still contains confounding from X

V2 = ____  # TODO: Y2 - Y2_hat (outcome residual)

# Regress residualized Y on RAW D (wrong!)
theta_naive = ____  # TODO: np.sum(D2 * V2) / np.sum(D2**2)

print(f"Method A - Naive ML (only Y residualized):  ${theta_naive:.2f}")


In [ ]:
# Method B: Full DML (residualize BOTH Y and D)
# This is CORRECT - creates Neyman orthogonality

W2 = ____  # TODO: D2 - D2_hat (treatment residual)

# Orthogonal score: regress residualized Y on residualized D
theta_full = ____  # TODO: np.sum(W2 * V2) / np.sum(W2**2)

print(f"Method B - Full DML (BOTH residualized):    ${theta_full:.2f}")
print(f"True ATE:                                   $2,500.00")
print("\n" + "="*60)
print("LESSON: You MUST residualize BOTH Y and D!")
print("         Using raw D leaves confounding in treatment.")
print("="*60)


## Part III: Comparison 2 - Why Sample Splitting Matters

Now we fix the method (Full DML) and compare:
- **Naive DML**: Train and predict on **SAME** data (no splitting)
- **Split DML**: Train on 1, predict on 2 (half-sample)
- **Cross-Fitted DML**: Both directions (full, correct!)

**Key insight**: Training on same data creates overfitting bias.

In [ ]:
# Method A: Naive DML (NO sample splitting - WRONG!)
# Using DEEP trees to maximize overfitting

l_full = RandomForestRegressor(
    n_estimators=30, max_depth=____, min_samples_leaf=____, random_state=42  # TODO: None, 1
)
l_full.fit(X, Y)
Y_hat_full = l_full.predict(X)  # Predict on SAME data!

m_full = RandomForestRegressor(
    n_estimators=30, max_depth=____, min_samples_leaf=____, random_state=42  # TODO: None, 1
)
m_full.fit(X, D)
D_hat_full = m_full.predict(X)  # Predict on SAME data!

# Compute residuals
V_full = Y - Y_hat_full
W_full = D - D_hat_full

# DML estimator
theta_no_split = ____  # TODO: np.sum(W_full * V_full) / np.sum(W_full**2)

print(f"Method A - No Splitting (overfitting):  ${theta_no_split:.2f}")


In [ ]:
# Method B & C: Cross-Fitted DML (CORRECT!)

# Split 2: Train on Sample 2, predict on Sample 1
l_model_2 = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
l_model_2.fit(X2, Y2)
Y1_hat = l_model_2.predict(X1)

m_model_2 = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
m_model_2.fit(X2, D2)
D1_hat = m_model_2.predict(X1)

# Compute residuals for Sample 1
V1 = Y1 - Y1_hat
W1 = D1 - D1_hat

# DML2: Pool all residuals and estimate (uses 100% of data!)
V_all = np.concatenate([V1, V2])
W_all = np.concatenate([W1, W2])

theta_cross_fitted = ____  # TODO: np.sum(W_all * V_all) / np.sum(W_all**2)

print(f"Method B - Half-Sample (wastes 50%):    ${theta_full:.2f}")
print(f"Method C - Cross-Fitted (uses 100%):    ${theta_cross_fitted:.2f}")


In [ ]:
# Evidence of overfitting
l_test = RandomForestRegressor(
    n_estimators=30, max_depth=None, min_samples_leaf=1, random_state=42
)
l_test.fit(X1, Y1)

r2_train = ____  # TODO: 1 - np.var(Y1 - l_test.predict(X1)) / np.var(Y1)
r2_test = ____   # TODO: 1 - np.var(Y2 - l_test.predict(X2)) / np.var(Y2)

print(f"\nEvidence of overfitting (outcome model):")
print(f"  R^2 on training data (same sample):  {r2_train:.3f}")
print(f"  R^2 on test data (new sample):       {r2_test:.3f}")
print(f"  Overfitting gap:                     {r2_train - r2_test:.3f}")
print("\n" + "="*60)
print("LESSON: You MUST use sample splitting!")
print("         Same-data predictions inflate R^2 -> bias in residuals")
print("="*60)


## Summary: All Methods Compared

In [ ]:
# Naive Mean for comparison
naive_ate = Y[D == 1].mean() - Y[D == 0].mean()

print("\n" + "="*60)
print("Final Comparison: All Methods")
print("="*60)
print(f"1. Naive Mean (no controls):           ${naive_ate:.2f}  ❌ Confounding")
print(f"2. Naive ML (single residualization):  ${theta_naive:.2f}  ❌ Orthogonality")
print(f"3. Naive DML (no splitting):           ${theta_no_split:.2f}  ❌ Overfitting")
print(f"4. Full DML (orthogonal + splitting):  ${theta_cross_fitted:.2f}  ✅ Correct!")
print(f"True ATE:                              $2,500.00")
print("="*60)
print("\nDML requires BOTH:")
print("  ✓ Orthogonalization: Residualize Y AND D")
print("  ✓ Sample Splitting:  Train on I_k^c, predict on I_k")


## Discussion Questions

**Q1**: Why is the Naive ML estimate (only residualizing Y) biased toward zero?

<details>
<summary>Click to reveal answer</summary>

**Answer**: Because treatment $D$ is still correlated with confounders $X$. The residualized outcome $V = Y - l(X)$ removes confounding from $Y$, but when we regress $V$ on raw $D$, we're still picking up the spurious correlation between $D$ and $X$. The correct approach is to residualize BOTH variables.

Mathematically, the bias comes from:
$$E[V \cdot D] = E[(Y-l(X)) \cdot D] = \underbrace{\theta \cdot E[D^2]}_{\text{what we want}} + \underbrace{E[\epsilon \cdot D]}_{\text{error}}$$

Since $D$ correlates with $X$ and $\epsilon$ correlates with $X$, the second term is non-zero unless we also residualize $D$.

</details>

---

**Q2**: Why does sample splitting matter if we already have orthogonalization?

<details>
<summary>Click to reveal answer</summary>

**Answer**: Orthogonalization alone doesn't solve the overfitting problem. When we train nuisance models on the same data we use for estimation, the predictions $\hat{l}(X_i)$ are too close to $Y_i$ (overfitting). This makes residuals $V_i = Y_i - \hat{l}(X_i)$ artificially small, biasing the final estimate.

**The intuition**: Imagine $\hat{l}(X)$ perfectly memorizes the training data. Then $V_i = Y_i - \hat{l}(X_i) \approx 0$ for all $i$, making the covariance $E[V \cdot W] \approx 0$, which biases $\hat{\theta}$ toward zero.

Sample splitting ensures $\hat{l}$ is trained on $I_k^c$ but evaluated on $I_k$, so predictions are independent of the estimation sample.

</details>

---

**Q3**: Why is the overfitting gap (R^2 train - R^2 test) evidence of the problem?

<details>
<summary>Click to reveal answer</summary>

**Answer**: The gap shows that the model performs much better on training data than test data. When we compute residuals $V = Y - \hat{l}(X)$ on the same data used for training, $\hat{l}(X)$ 'knows' $Y$ and predictions are too good.

**Consequences**:
1. Residuals $V_i$ are artificially shrunk toward zero
2. Covariance $E[V \cdot W]$ is biased
3. Final estimate $\hat{\theta}$ is inconsistent

This is why we see the gap of ~0.20 in the exercise -- the model has 'memorized' the training data, leading to over-optimistic predictions and biased residuals.

</details>

---